# Homework 4 — Evaluating keyword, vector and hybrid search

Spec: [cohorts/2026/homework/04-evaluation/homework.md](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/homework/04-evaluation/homework.md)

Homework 2 built three searches over the 72 course lesson pages (commit `8c1834d`); homework 4 measures which is best.
This notebook rebuilds the homework 2 pieces it needs (ONNX embedder, chunks, RRF) and then evaluates them.

Setup: `uv run python download.py` fetches the ONNX `all-MiniLM-L6-v2` into `models/` (gitignored).
`download.py` and `embedder.py` are the course's helpers from
[02-vector-search/embed](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/02-vector-search/embed).

In [1]:
import json

import numpy as np
import pandas as pd
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch
from pydantic import BaseModel
from tqdm.auto import tqdm

from embedder import Embedder
from eval_utils import evaluate, llm_structured

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents)

72

## Q1. Generating questions

Same structured-output approach as the module: a `Questions` model, a JSON user prompt with the page's
`filename` and `content`, instructions that ask for reworded questions so the test isn't trivially easy.

In [2]:
class Questions(BaseModel):
    questions: list[str]


data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

first_pages = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]
by_name = {d["filename"]: d for d in documents}

generated, input_tokens = [], []
for name in first_pages:
    doc = by_name[name]
    user_prompt = json.dumps({"filename": doc["filename"], "content": doc["content"]})
    result, usage = llm_structured(data_gen_instructions, user_prompt, Questions)
    input_tokens.append(usage.prompt_tokens)
    generated.extend({"question": q, "filename": name} for q in result.questions)

print("input tokens per call:", input_tokens, "| average:", round(np.mean(input_tokens)))
generated[:5]

input tokens per call: [1150, 1416, 1883] | average: 1483


[{'question': "What is the main goal of the Retrieval-Augmented Generation (RAG) system we'll build in this module?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'How does the course treat large language models when using them in the projects?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why do we need to retrieve documents before sending a query to the LLM?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the two main parts of this module and what does each cover?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': "Where can I find the instructor's contact or social media links mentioned in the intro?",
  'filename': '01-agentic-rag/lessons/01-intro.md'}]

## The full ground truth

The course's 360 prepared questions (5 per page). Note: the homework's `cohorts/2026/...` URL now 404s; the file lives at [`04-evaluation/ground-truth.csv`](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/04-evaluation/ground-truth.csv).

In [3]:
ground_truth = pd.read_csv("data/ground-truth-lessons.csv").to_dict(orient="records")
len(ground_truth), ground_truth[0]

(360,
 {'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'})

## Searching the chunks (from homework 2)

- **Text**: minsearch `Index` over chunk `content`.
- **Vector**: ONNX `all-MiniLM-L6-v2` (identical vectors to sentence-transformers, ~30× lighter install).
- **Hybrid**: Reciprocal Rank Fusion. Each document scores `Σ 1/(k + rank)` over the lists it appears in, so agreement
  between retrievers is rewarded without having to compare their incompatible raw scores.

In [4]:
chunks = chunk_documents(documents, size=2000, step=1000)

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

embedder = Embedder()
print("homework 2 Q1 — v[0] for the ANN query:",
      round(float(embedder.encode("How does approximate nearest neighbor search work?")[0]), 2))

X = np.vstack([embedder.encode_batch([c["content"] for c in chunks[i:i + 32]]) for i in tqdm(range(0, len(chunks), 32))])
vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)


def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    return vector_index.search(embedder.encode(query), num_results=num_results)


def rrf(result_lists, k=60, num_results=5):
    scores, docs = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    return rrf([text_search(query, num_results=10), vector_search(query, num_results=10)], k=k)


len(chunks), X.shape

homework 2 Q1 — v[0] for the ANN query: -0.02


  0%|          | 0/10 [00:00<?, ?it/s]

(295, (295, 384))

## Q2 & Q3. First result for the first question

In [5]:
q = ground_truth[0]["question"]
print("question:", q)
print("label:   ", ground_truth[0]["filename"])
print("Q2 text:  ", text_search(q)[0]["filename"])
print("Q3 vector:", vector_search(q)[0]["filename"])

question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
label:    01-agentic-rag/lessons/01-intro.md
Q2 text:   01-agentic-rag/lessons/03-rag.md
Q3 vector: 01-agentic-rag/lessons/01-intro.md


## Evaluation

A hit is a returned chunk whose `filename` matches the question's page. The relevance rule is the only change from
the lesson's FAQ evaluation (which compared record `id`s); `evaluate` takes it as a parameter.

In [6]:
def is_relevant(q, doc):
    return doc["filename"] == q["filename"]


results = {
    "text": evaluate(ground_truth, text_search, is_relevant),
    "vector": evaluate(ground_truth, vector_search, is_relevant),
    "hybrid (k=60)": evaluate(ground_truth, hybrid_search, is_relevant),
}
pd.DataFrame(results).T.round(3)

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

,hit_rate,mrr
text,0.758,0.594
vector,0.725,0.549
hybrid (k=60),0.836,0.638


## Q6. Tuning `k` in RRF

In [7]:
k_results = {k: evaluate(ground_truth, lambda query: hybrid_search(query, k=k), is_relevant, progress=False)
             for k in [1, 50, 100, 200]}
pd.DataFrame(k_results).T.round(4)

,hit_rate,mrr
1,0.8389,0.6482
50,0.8361,0.6379
100,0.8361,0.6379
200,0.8361,0.6379


## Extra — the embedder only reads the first 128 tokens of each chunk

The ONNX tokenizer ships with `truncation: max_length=128`. A 2000-character chunk is ~530 tokens (median),
so **~75% of every chunk is silently ignored** when it's embedded. Text search reads the whole chunk.
That alone could explain part of the gap between text and vector.

Two fixes to test, everything else fixed:
1. raise truncation to 256 tokens (what sentence-transformers uses for this model by default);
2. smaller chunks (500 chars, step 250) that fit inside 128 tokens.

In [8]:
lengths = [len(embedder.tokenizer.encode(c["content"]).ids) for c in chunks[:1]]
embedder.tokenizer.no_truncation()
token_counts = np.array([len(embedder.tokenizer.encode(c["content"]).ids) for c in chunks])
print(f"tokens per 2000-char chunk: median {np.median(token_counts):.0f}, max {token_counts.max()}")


def vector_eval(chunk_list, max_tokens):
    embedder.tokenizer.enable_truncation(max_tokens)
    vecs = np.vstack([embedder.encode_batch([c["content"] for c in chunk_list[i:i + 32]])
                      for i in range(0, len(chunk_list), 32)])
    index = VectorSearch(keyword_fields=["filename"]).fit(vecs, chunk_list)

    def search(query, num_results=5):
        return index.search(embedder.encode(query), num_results=num_results)

    return evaluate(ground_truth, search, is_relevant, progress=False)


small_chunks = chunk_documents(documents, size=500, step=250)
experiments = {
    "2000-char chunks, 128 tokens (homework)": vector_eval(chunks, 128),
    "2000-char chunks, 256 tokens": vector_eval(chunks, 256),
    f"500-char chunks ({len(small_chunks)}), 128 tokens": vector_eval(small_chunks, 128),
}
embedder.tokenizer.enable_truncation(128)   # restore the shipped default
pd.DataFrame(experiments).T.round(3)

tokens per 2000-char chunk: median 528, max 791


,hit_rate,mrr
"2000-char chunks, 128 tokens (homework)",0.725,0.549
"2000-char chunks, 256 tokens",0.808,0.636
"500-char chunks (1296), 128 tokens",0.828,0.665


## Answers

| Q | Measured | Answer |
|---|----------|--------|
| 1 | avg input tokens 1,483 (1,150 / 1,416 / 1,883) | **1400** |
| 2 | text top-1: `01-agentic-rag/lessons/03-rag.md` | **03-rag.md** |
| 3 | vector top-1: `01-agentic-rag/lessons/01-intro.md` | **01-intro.md** |
| 4 | text hit rate 0.758 | **0.76** |
| 5 | vector MRR 0.549 | **0.55** |
| 6 | MRR 0.648 at k=1, 0.638 at 50/100/200 | **1** |

## Findings

1. **Hybrid wins** (hit rate 0.84 vs 0.76 text / 0.73 vector). Text and vector miss *different* questions, and RRF
   rewards chunks either one ranks highly.
2. **`k` barely matters here** (0.648 vs 0.638). Small `k` sharpens the reward for rank 1, which helps slightly when the two
   lists already agree on the top chunk. Not a big lever on this data.
3. **"Text beats vector" is partly a truncation artifact.** The ONNX tokenizer truncates at 128 tokens, while chunks are
   ~530 tokens. Fix only that and vector MRR goes **0.549 → 0.636**, above text (0.594). Smaller chunks that fit the
   window give **0.665**. Before comparing retrieval methods, check that each one actually sees the whole document.
   An embedding model's max sequence length is a chunking constraint.
4. One question is not a benchmark: Q2/Q3 show text and vector disagreeing on the first question, but only the full
   360-question evaluation shows which method is better overall.